PRACTICE EXERCISE — Network Triage Sweep (10 min)

Scenario: pretend a ticket just landed — "intermittent connectivity issues reported, please do a quick health check." Run these five commands in order, and after each one write a one-line verdict (`PASS` / `FAIL` / `N/A` + why) before moving to the next — don't just run them all and eyeball at the end, the point is practicing the triage order.

**What to check at each step:**

1. `ip -br addr` — is `eth0` `UP` with a valid IP?
2. `ip link` (full, not `-br` — you need the MTU column) — is MTU `1500`? Anything else would be your first red flag.
3. `ip route` — is there a `default via ...` line at all?
4. Gateway ping — record the result either way, but **don't conclude anything from it alone** yet (you already know why from last session).
5. `ping -c 4 1.1.1.1` — real external reachability test.

**Your task:** after running all five, write me a short verdict — "network healthy" or "network has a problem" — and back it up with which specific output told you that, not just "it worked." If step 4 fails but step 5 passes, I want to see you explain _why that's still a healthy verdict_ in your own words, not just cite the gotcha.

In [ ]:
dennie@X390:~/linux-tut$ ip -br addr
lo               UNKNOWN        127.0.0.1/8 10.255.255.254/32 ::1/128
eth0             UP             172.30.240.133/20 fe80::215:5dff:feca:59a4/64

This shows ALL interfaces and their IP addresses(IPv4, IPv6) on this machine, with `-br` flag for brief reference - only the name, state (Up or Down) and address. `eth0` says everything for a first glance. So, the real network interface `eth0` is UP - administratively up, and `/20` says the first 20 bits are the network portion, fixed, shared by everyone on this subnet, the remaining 12 bits are the host portion.

In [ ]:
dennie@X390:~/linux-tut$ ip link
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN mode DEFAULT group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP mode DEFAULT group default qlen 1000
    link/ether 00:15:5d:ca:59:a4 brd ff:ff:ff:ff:ff:ff

It's the link/device layer: is this interface administratively on or off, what's its MTU, what's its MAC. So, it's up, maximum tramsmission unit is 1500 bytes - the largest packet size this interface will send without fragmenting, standard for Ethernet, and MAC address `link/ether 00:15:5d:ca:59:a4`

In [ ]:
dennie@X390:~/linux-tut$ ip route
default via 172.30.240.1 dev eth0 proto kernel
172.30.240.0/20 dev eth0 proto kernel scope link src 172.30.240.133

There is a default gateway (route), it's present here under `default via 172.30.240.1 dev eth0 proto kernel`. So, the box is not isolated of anything off-subnet even though own IP/interface look perfectly healthy. Anything not matched by a more specific line below (`172.30.240.0/20 dev eth0...`) gets sent to 172.30.240.1, the gateway out through `eth0` interface.

In [ ]:
dennie@X390:~/linux-tut$ ping -c 4 172.30.240.1
PING 172.30.240.1 (172.30.240.1) 56(84) bytes of data.

--- 172.30.240.1 ping statistics ---
4 packets transmitted, 0 received, 100% packet loss, time 3018ms

I sent ICMP (Internet Control Message Protocol) echo request packets to my own gateway, and got 100% paclet loss, zero replies. My own gateway does not reply ICMP echo requests addressed to itself. I need to sent ping request to an external host to make sure it goes well.

In [ ]:
dennie@X390:~/linux-tut$ ping -c 4 1.1.1.1
PING 1.1.1.1 (1.1.1.1) 56(84) bytes of data.
64 bytes from 1.1.1.1: icmp_seq=1 ttl=56 time=229 ms
64 bytes from 1.1.1.1: icmp_seq=2 ttl=56 time=91.9 ms
64 bytes from 1.1.1.1: icmp_seq=3 ttl=56 time=126 ms
64 bytes from 1.1.1.1: icmp_seq=4 ttl=56 time=70.4 ms

--- 1.1.1.1 ping statistics ---
4 packets transmitted, 4 received, 0% packet loss, time 2997ms
rtt min/avg/max/mdev = 70.427/129.309/228.794/60.769 ms

Sending an echo request to the real external host confirms the gateway works allowing passing through itself this echo request. WSL's virutal dateway silently drops ICMP addressed directly to it while still forwarding everything through it perfectly. In a real ticket I would check wheather traffic actually passes through it with an external host ping. The gateway is healthy as well as the network. icmp_seq=1 is 229 ms, mdev=60.769ms probably about route/ARP-cache warm-up.

Graded the whole sweep — this is strong work, walking through it in order like you did is exactly the right habit.

**Step-by-step accuracy:**

1. `ip -br addr` → correctly read `eth0 UP` with valid IP, and you tied the `/20` back to network/host bit split from the subnetting vocab.
2. `ip link` → correctly caught `mtu 1500` (standard, no red flag) and the MAC — and you used the full form, not `-br`, exactly as the exercise called for.
3. `ip route` → correctly identified the default route and explained the fallback logic (unmatched traffic → gateway) in your own words, not just quoting the lesson.
4. Gateway ping → this is the one that actually tests understanding, and you got it right: 100% loss noted, but you correctly refused to conclude "gateway down" from that alone and moved to an external test instead.
5. `1.1.1.1` ping → 0% loss, and you correctly reasoned the WSL virtual gateway forwards traffic fine while just not answering ICMP addressed to itself — that's the actual takeaway, not just a memorized gotcha.

**Verdict:** you delivered it — "the gateway is healthy as well as the network" — backed by the right evidence (external ping succeeded, gateway ping alone doesn't disprove that). Only nitpick: it's folded into the last paragraph rather than stated as its own line — fine for practice, but in a real ticket note or handoff, put the verdict first, evidence after, so whoever reads it next doesn't have to hunt for your conclusion.